In [ ]:
# ================================================================================
# CONFIGURATION FOR KAGGLE
# ================================================================================
import os

# Paths - Thay đổi theo cấu trúc dữ liệu của bạn
DATA_DIR = "/kaggle/input/datasets/hngphm007/mri-2d-dataset/processed_oasis_2d_jpeg"
CHECKPOINT_DIR = "/kaggle/working/checkpoints"
LOG_DIR = "/kaggle/working/logs"

# Data Configuration
VIEW = "axial"  # "axial", "coronal", "sagittal", or "all"
NUM_SLICES = 80  # Auto-computed from view if None

# Training Configuration
SEED = 42
NUM_FOLDS = 5
BATCH_SIZE = 16
NUM_EPOCHS = 60
LEARNING_RATE = 1e-3
WEIGHT_DECAY = 1e-4
USE_AMP = True  # Mixed precision training
USE_CLASS_WEIGHTS = True

# Model Configuration
MODEL_CHOICE = "densenet"  # "densenet" or "mobilenet"
EMBED_DIM = 256
NUM_CLASSES = 2

# Scheduler & Early Stopping
SCHEDULER = "cosine"  # "cosine", "step", or None
EARLY_STOPPING_PATIENCE = 15

# Logging
USE_TENSORBOARD = True
USE_WANDB = False
WANDB_PROJECT = "alzheimer-classification"
EXP_NAME = "axial_densenet"

# Create output directories
os.makedirs(CHECKPOINT_DIR, exist_ok=True)
os.makedirs(LOG_DIR, exist_ok=True)

print(f"DATA_DIR: {DATA_DIR}")
print(f"CHECKPOINT_DIR: {CHECKPOINT_DIR}")
print(f"VIEW: {VIEW}")
print(f"NUM_SLICES: {NUM_SLICES}")
print(f"MODEL: {MODEL_CHOICE.upper()}")


In [ ]:
# ================================================================================
# IMPORTS
# ================================================================================
import os
import sys
import random
import json
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision.transforms import ColorJitter, Compose, Resize, ToTensor, Normalize, RandomAffine
from torch.utils.data import Dataset, DataLoader
from tqdm import tqdm
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score, confusion_matrix
)
from PIL import Image

try:
    from torch.utils.tensorboard import SummaryWriter
    HAS_TENSORBOARD = True
except ImportError:
    HAS_TENSORBOARD = False

try:
    import wandb
    HAS_WANDB = True
except ImportError:
    HAS_WANDB = False

print("[INFO] All imports loaded successfully")


In [ ]:
class OASIS2DDataset(Dataset):
    """
    Dataset for OASIS MRI classification (Alzheimer vs Normal) from JPEG images.
    Compatible with both pre-processed JPEG folder structures.
    """
    
    CLASSES = {'normal': 0, 'alzheimer': 1, 'cn': 0, 'ad': 1}

    def __init__(self, root=None, transform=None, view=None, split='train', fold=0):
        """
        Args:
            root: Root directory path
            transform: Transforms to apply to images
            view: 'axial', 'coronal', 'sagittal', or 'All'
            split: 'train', 'val', 'test'
            fold: Fold number (0-4)
        """
        self.transform = transform
        self.image_paths = []
        self.labels = []
        self.id_patient = []
        self.patients = {}  # patient_id -> label
        self.view = view or "All"
        
        if root is None:
            root = "/kaggle/working"

        # Try to load from fold structure first
        self._load_from_processed_structure(root)
        
        if not self.image_paths:
            # Fallback to flat structure
            self._load_from_flat_structure(root)

    def _load_from_processed_structure(self, root):
        """Load from processed_oasis_2d_jpeg structure with fold_{fold}/split/class/"""
        views_to_check = ['axial', 'coronal', 'sagittal'] if self.view == "All" else [self.view]
        
        for view in views_to_check:
            view_root = os.path.join(root, 'processed_oasis_2d_jpeg', view, 'fold_0')
            
            if not os.path.exists(view_root):
                continue
                
            for split_dir in ['train', 'val', 'test']:
                split_path = os.path.join(view_root, split_dir)
                if not os.path.exists(split_path):
                    continue
                    
                for class_name in ['normal', 'nonnormal']:
                    class_path = os.path.join(split_path, class_name)
                    if not os.path.exists(class_path):
                        continue
                    
                    label = 0 if class_name == 'normal' else 1
                    
                    for file in sorted(os.listdir(class_path)):
                        if file.lower().endswith(('.jpg', '.jpeg', '.png')):
                            path = os.path.join(class_path, file)
                            # Extract subject_id from filename
                            parts = file.replace('.jpg', '').replace('.png', '').split('_')
                            if len(parts) >= 3:
                                subject_id = '_'.join(parts[:3])  # OAS1_0001_MR1
                                
                                self.image_paths.append(path)
                                self.labels.append(label)
                                self.id_patient.append(subject_id)
                                
                                if subject_id not in self.patients:
                                    self.patients[subject_id] = label

    def _load_from_flat_structure(self, root):
        """Fallback: Load from flat CN/AD structure (for Kaggle)"""
        for label, category in enumerate(['CN', 'AD']):
            class_dir = os.path.join(root, category)
            if not os.path.exists(class_dir):
                continue
                
            for subject_id in os.listdir(class_dir):
                subject_path = os.path.join(class_dir, subject_id)
                if not os.path.isdir(subject_path):
                    continue
                
                patient_name = os.path.basename(subject_id)
                self.patients[patient_name] = label
                
                for file in os.listdir(subject_path):
                    if file.lower().endswith(('.png', '.jpg', '.jpeg')):
                        path = os.path.join(subject_path, file)
                        
                        if self.view == "All":
                            include = True
                        elif self.view.lower() == "axial" and "axial" in file.lower():
                            include = True
                        elif self.view.lower() == "coronal" and "coronal" in file.lower():
                            include = True
                        elif self.view.lower() == "sagittal" and "sagittal" in file.lower():
                            include = True
                        else:
                            include = False
                        
                        if include:
                            self.image_paths.append(path)
                            self.labels.append(label)
                            self.id_patient.append(patient_name)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, index):
        image_path = self.image_paths[index]
        label = self.labels[index]
        patient_id = self.id_patient[index]
        
        image = Image.open(image_path).convert("RGB")
        
        if self.transform:
            image = self.transform(image)
        
        return image, label, patient_id


# Model.py

In [ ]:
class MyEfficentNet(nn.Module):
    def __init__(self, num_classes=2):
        super().__init__()
        self.backbone = efficientnet_b0(weights=EfficientNet_B0_Weights.DEFAULT)
        self.backbone.classifier = nn.Identity()
        self.classifier = nn.Sequential(
            nn.Dropout(p=0.5, inplace=True),
            nn.Linear(1280, num_classes),
        )
    
    def forward(self, x):
        return self.classifier(self.backbone(x))


In [28]:
class MyResNet50(nn.Module):
    def __init__(self, num_classes=10):
        super().__init__()
        self.backbone = resnet50(weights=ResNet50_Weights.DEFAULT)
        self.backbone.fc = nn.Identity()  # bỏ fc gốc

        self.classifier = nn.Sequential(
            nn.Linear(2048, 1024),
            nn.Linear(1024, num_classes)
        )

    def forward(self, x):
        x = self.backbone(x)  # ra (batch, 2048)
        x = self.classifier(x)
        return x

In [29]:
class MyConvNeXt(nn.Module):
    def __init__(self, num_classes=10):
        super().__init__()
        self.backbone = convnext_tiny(weights=ConvNeXt_Tiny_Weights.DEFAULT)
        
        # bỏ classifier gốc
        self.backbone.classifier = nn.Identity()

        self.classifier = nn.Sequential(
            nn.LayerNorm(768),
            nn.Linear(768, num_classes)
        )

    def forward(self, x):
        x = self.backbone(x)   # (B, 768, 1, 1)
        x = torch.flatten(x, 1)  
        x = self.classifier(x)
        return x

In [ ]:
class MyDenseNet121(nn.Module):
    def __init__(self, num_classes=2, pretrained=True, use_attention=False, embed_dim=256):
        super().__init__()

        if pretrained:
            weights = DenseNet121_Weights.IMAGENET1K_V1
        else:
            weights = None

        self.backbone = densenet121(weights=weights)
        in_features = self.backbone.classifier.in_features
        self.backbone.classifier = nn.Identity()
        
        self.use_attention = use_attention
        
        if use_attention:
            # Attention-based aggregation
            self.proj = nn.Linear(in_features, embed_dim)
            self.attn = nn.MultiheadAttention(
                embed_dim=embed_dim,
                num_heads=8,
                batch_first=True,
            )
            self.classifier = nn.Sequential(
                nn.LayerNorm(embed_dim),
                nn.Linear(embed_dim, num_classes),
            )
        else:
            # Simple classifier
            self.classifier = nn.Sequential(
                nn.Linear(in_features, 256),
                nn.ReLU(inplace=True),
                nn.Dropout(0.3),
                nn.Linear(256, num_classes)
            )

    def forward(self, x):
        x = self.backbone(x)
        
        if self.use_attention:
            x = self.proj(x)
            x = x.unsqueeze(0) if x.dim() == 1 else x
            if x.dim() == 2:
                x = x.unsqueeze(1)
            attn_out, _ = self.attn(x, x, x)
            x = (x + attn_out).mean(dim=1)
        
        x = self.classifier(x)
        return x


In [ ]:
class MyMobileNetMultiAttention(nn.Module):
    """Lightweight model suitable for Kaggle with attention mechanism"""
    def __init__(self, num_classes=2, num_slices=80, embed_dim=256):
        super().__init__()
        from torchvision.models import mobilenet_v3_small, MobileNet_V3_Small_Weights
        
        base_model = mobilenet_v3_small(weights=MobileNet_V3_Small_Weights.DEFAULT)
        base_model.features[0][0] = nn.Conv2d(3, 16, 3, stride=2, padding=1, bias=False)
        
        self.backbone = base_model.features
        self.avgpool = nn.AdaptiveAvgPool2d((1, 1))
        
        self.feature_dim = base_model.classifier[0].in_features
        self.proj = nn.Linear(self.feature_dim, embed_dim)
        
        self.attn = nn.MultiheadAttention(embed_dim, num_heads=4, batch_first=True)
        
        self.classifier = nn.Sequential(
            nn.LayerNorm(embed_dim),
            nn.Dropout(p=0.5),
            nn.Linear(embed_dim, num_classes)
        )
        
    def forward(self, x):
        # For compatibility with single image input (not sequences)
        if x.dim() == 4:  # (B, C, H, W)
            features = self.backbone(x)
            features = self.avgpool(features).flatten(1)
            features = self.proj(features)
        else:  # (B, S, C, H, W) - sequence of slices
            B, S, C, H, W = x.shape
            x = x.view(B * S, C, H, W)
            features = self.backbone(x)
            features = self.avgpool(features).flatten(1)
            features = features.view(B, S, self.feature_dim)
            features = self.proj(features)
            attn_out, _ = self.attn(features, features, features)
            features = (features + attn_out).mean(dim=1)
        
        logits = self.classifier(features)
        return logits


In [32]:
class GradCAM:
    def __init__(self, model, target_layer):
        self.model = model
        self.target_layer = target_layer

        self.activations = None
        self.gradients = None

        # đăng ký hook
        self.fwd_handle = target_layer.register_forward_hook(self._save_activation)
        self.bwd_handle = target_layer.register_full_backward_hook(self._save_gradient)

    def _save_activation(self, module, input, output):
        self.activations = output

    def _save_gradient(self, module, grad_input, grad_output):
        self.gradients = grad_output[0]

    def generate(self, logits, class_idx, input_size):
        self.model.zero_grad()

        one_hot = torch.zeros_like(logits)
        one_hot[:, class_idx] = 1
        logits.backward(one_hot)

        weights = torch.mean(self.gradients, dim=(2, 3), keepdim=True)
        cam = torch.sum(weights * self.activations, dim=1)
        cam = torch.relu(cam).squeeze()
        cam = cam - cam.min()
        cam = cam / (cam.max() + 1e-8)
        cam = cam.detach().cpu().numpy()

        H, W = input_size
        cam = cv2.resize(cam, (W, H))
        return cam

# Train.py

In [ ]:
import random

def get_args():
    parser = ArgumentParser(description="CNN training")
    parser.add_argument("--seed", type=int, default=42)
    parser.add_argument("--root", "-r", type=str, default=DATA_PROCESSED, help="Root of the dataset")
    parser.add_argument("--folds", "-f", type=int, default=NUM_FOLDS, help="Number of folds")
    parser.add_argument("--epochs", "-e", type=int, default=NUM_EPOCHS, help="Number of epochs")
    parser.add_argument("--weight_decay", type=float, default=WEIGHT_DECAY)
    parser.add_argument("--early_stopping", "-s", type=int, default=EARLY_STOPPING)
    parser.add_argument("--batch_size", "-b", type=int, default=BATCH_SIZE, help="Batch size")
    parser.add_argument("--learning_rate", type=float, default=LEARNING_RATE)
    parser.add_argument("--image_size", "-i", type=int, default=IMAGE_SIZE, help="Image size")
    parser.add_argument("--logging", "-l", type=str, default=f"{OUTPUT}/tensorboard")
    parser.add_argument("--trained_models", "-t", type=str, default=f"{OUTPUT}/trained_models")
    parser.add_argument("--checkpoint", "-c", type=str, default=None)
    parser.add_argument("--model", type=str, default="densenet", choices=["densenet", "mobilenet"])
    args = parser.parse_args(args=[])
    return args

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    os.environ['PYTHONHASHSEED'] = str(seed)


In [34]:
def visualize_cam(patient_id, model, dataset, device, save_dir):

    model.eval()

    target_layer = model.backbone.features[-1]
    camger = GradCAM(model, target_layer)

    # bucket theo view
    frames_dict = {}

    patient_indices = [
        i for i, pid in enumerate(dataset.id_patient)
        if pid == patient_id
    ]

    for idx in patient_indices:

        # Lấy thông tin slice
        img_path = dataset.image_paths[idx]
        filename = os.path.basename(img_path)
        
        # Xác định view từ tên file (axial, coronal, sagittal)
        if 'axial' in filename.lower():
            view_name = 'axial'
        elif 'coronal' in filename.lower():
            view_name = 'coronal'
        elif 'sagittal' in filename.lower():
            view_name = 'sagittal'
        else:
            view_name = 'unknown'
            
        img, label, _ = dataset[idx]
        x = img.unsqueeze(0).to(device)

        logits = model(x)
        prob = torch.softmax(logits, dim=1)[0,1].item()

        cam = camger.generate(logits, class_idx=1, input_size=(x.shape[2], x.shape[3]))

        # unnormalize
        mean = torch.tensor([0.485,0.456,0.406]).view(3,1,1)
        std  = torch.tensor([0.229,0.224,0.225]).view(3,1,1)
        img_vis = img * std + mean
        img_vis = img_vis.permute(1,2,0).cpu().numpy()

        heatmap = cv2.applyColorMap(np.uint8(cam*255), cv2.COLORMAP_JET)
        heatmap = cv2.cvtColor(heatmap, cv2.COLOR_BGR2RGB)/255.0

        overlay = 0.6*img_vis + 0.4*heatmap
        overlay = np.clip(overlay,0,1)

        frame = (overlay*255).astype(np.uint8)

        text = f"{view_name.upper()} | Score: {prob:.4f}"
        cv2.putText(
            frame,
            text,
            (20,30),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.8,
            (255,255,255),
            2,
            cv2.LINE_AA
        )

        # tạo bucket nếu chưa tồn tại
        if view_name not in frames_dict:
            frames_dict[view_name] = []

        frames_dict[view_name].append(frame)

    # save mỗi view thành 1 GIF
    for view_name, frames in frames_dict.items():
        save_path = f"{save_dir}/{patient_id}_{view_name}.gif"
        imageio.mimsave(save_path, frames, fps=3)

In [35]:
def plot_confusion_matrix(writer, cm, class_names, epoch):

    cm_raw = cm.copy()

    cm_norm = cm.astype('float') / cm.sum(axis=1, keepdims=True)
    cm_norm = np.nan_to_num(cm_norm)
    cm_norm = np.round(cm_norm, 2)

    plt.style.use("seaborn-v0_8-white")
    figure = plt.figure(figsize=(8, 8), dpi=200)

    ax = sns.heatmap(
        cm_norm,
        annot=False,
        cmap="Blues",
        cbar=True,
        square=True,
        linewidths=1,
        linecolor='gray'
    )

    ax.set_xticklabels(class_names, rotation=45, fontsize=12)
    ax.set_yticklabels(class_names, rotation=0, fontsize=12)

    ax.set_ylabel("True Label", fontsize=14)
    ax.set_xlabel("Predicted Label", fontsize=14)
    ax.set_title("Confusion Matrix (Normalized)", fontsize=16, pad=20)

    threshold = cm_norm.max() / 2.0

    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]):
            text_color = "white" if cm_norm[i, j] > threshold else "black"
            ax.text(
                j + 0.5,
                i + 0.5,
                f"{cm_raw[i, j]}\n({cm_norm[i, j]:.2f})",
                ha="center",
                va="center",
                color=text_color,
                fontsize=12,
                fontweight="bold"
            )

    plt.tight_layout()

    writer.add_figure("Confusion_Matrix", figure, epoch)
    plt.show()
    plt.close(figure)

In [36]:
def plot_metrics(
    writer,
    train_losses,
    val_acc,
    val_auc,
    val_labels,
    val_probs,
    class_names,
    epoch,
    tag="",
    save_dir=None
):
    """
    Vẽ 5 metric trong 1 figure:

    Row 1:
        - Train Loss
        - Val Accuracy
        - Val AUC

    Row 2:
        - Confusion Matrix
        - ROC Curve
        - (empty)
    """

    epochs = list(range(1, len(train_losses) + 1))

    fig, axes = plt.subplots(2, 3, figsize=(18, 10), dpi=200)

    # =======================
    # ROW 1
    # =======================

    # Train Loss
    axes[0, 0].plot(epochs, train_losses, marker='o', linewidth=2)
    axes[0, 0].set_title(f"{tag} - Train Loss")
    axes[0, 0].set_xlabel("Epoch")
    axes[0, 0].set_ylabel("Loss")
    axes[0, 0].grid(True)

    # Val Accuracy
    axes[0, 1].plot(epochs, val_acc, marker='s', linewidth=2)
    axes[0, 1].set_title(f"{tag} - Val Accuracy")
    axes[0, 1].set_xlabel("Epoch")
    axes[0, 1].set_ylabel("Accuracy")
    axes[0, 1].set_ylim(0, 1)
    axes[0, 1].grid(True)

    # Val AUC
    axes[0, 2].plot(epochs, val_auc, marker='^', linewidth=2)
    axes[0, 2].set_title(f"{tag} - Val AUC")
    axes[0, 2].set_xlabel("Epoch")
    axes[0, 2].set_ylabel("AUC")
    axes[0, 2].set_ylim(0, 1)
    axes[0, 2].grid(True)

    # =======================
    # ROW 2
    # =======================

    # Confusion Matrix (best threshold = 0.5)
    preds_bin = [1 if p >= 0.5 else 0 for p in val_probs]
    cm = confusion_matrix(val_labels, preds_bin)

    cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)
    cm_norm = np.nan_to_num(cm_norm)
    cm_norm = np.round(cm_norm, 2)

    sns.heatmap(
        cm_norm,
        annot=False,
        cmap="Blues",
        cbar=True,
        square=True,
        linewidths=1,
        linecolor='gray',
        ax=axes[1, 0]
    )

    axes[1, 0].set_title(f"{tag} - Confusion Matrix")
    axes[1, 0].set_xlabel("Predicted")
    axes[1, 0].set_ylabel("True")
    axes[1, 0].set_xticklabels(class_names, rotation=45)
    axes[1, 0].set_yticklabels(class_names, rotation=0)

    threshold = cm_norm.max() / 2.0

    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]):
            color = "white" if cm_norm[i, j] > threshold else "black"
            axes[1, 0].text(
                j + 0.5,
                i + 0.5,
                f"{cm[i,j]}\n({cm_norm[i,j]:.2f})",
                ha="center",
                va="center",
                color=color,
                fontsize=11,
                fontweight="bold"
            )

    # ROC Curve
    try:
        fpr, tpr, _ = roc_curve(val_labels, val_probs)
        auc_score = roc_auc_score(val_labels, val_probs)
        axes[1, 1].plot(fpr, tpr, linewidth=2, label=f"AUC = {auc_score:.4f}")
    except ValueError:
        axes[1, 1].text(0.5, 0.5, "AUC N/A", ha='center', va='center')

    axes[1, 1].plot([0, 1], [0, 1], linestyle="--")
    axes[1, 1].set_title(f"{tag} - ROC Curve")
    axes[1, 1].set_xlabel("FPR")
    axes[1, 1].set_ylabel("TPR")
    axes[1, 1].legend()
    axes[1, 1].grid(True)

    # Hide last subplot (2,3)
    axes[1, 2].axis("off")

    plt.tight_layout()

    if save_dir is not None:
        os.makedirs(save_dir, exist_ok=True)
        fname = os.path.join(save_dir, f"{tag}_all_metrics.png".replace(" ", "_"))
        fig.savefig(fname, dpi=200)

    writer.add_figure(f"{tag}/All_Metrics", fig, epoch)

    plt.show()
    plt.close(fig)

In [37]:
def collect_fold_cams(val_patients, model, full_dataset, device, save_dir):

    model.eval()

    target_layer = model.backbone.features[-1]
    camger = GradCAM(model, target_layer)

    views = ["axial", "coronal", "sagittal"] if VIEWS_TO_USE == "All" else [VIEWS_TO_USE]
    val_patients_set = set(val_patients)

    # Gom tất cả indices của val patients
    val_indices = [
        i for i, pid in enumerate(full_dataset.id_patient)
        if pid in val_patients_set
    ]

    # Batch forward để lấy scores (không cần grad)
    all_imgs = torch.stack([full_dataset[i][0] for i in val_indices]).to(device)

    chunk_size = 64
    all_probs = []
    with torch.no_grad():
        for i in range(0, len(all_imgs), chunk_size):
            chunk = all_imgs[i:i + chunk_size]
            logits = model(chunk)
            probs = torch.softmax(logits, dim=1)[:, 1].cpu().tolist()
            all_probs.extend(probs)

    mean = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
    std  = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)

    for local_idx, global_idx in enumerate(val_indices):

        pid = full_dataset.id_patient[global_idx]
        image_path = full_dataset.image_paths[global_idx]

        view_name = next((v for v in views if v in image_path.lower()), None)
        if view_name is None:
            continue

        # Tạo folder riêng cho từng patient
        patient_dir = os.path.join(save_dir, pid)
        os.makedirs(patient_dir, exist_ok=True)

        x = all_imgs[local_idx].unsqueeze(0)

        logits = model(x)
        prob = all_probs[local_idx]

        cam = camger.generate(logits, class_idx=1, input_size=(x.shape[2], x.shape[3]))

        # Unnormalize ảnh gốc
        img = all_imgs[local_idx].cpu()
        img_vis = (img * std + mean).permute(1, 2, 0).numpy()

        heatmap = cv2.applyColorMap(np.uint8(cam * 255), cv2.COLORMAP_JET)
        heatmap = cv2.cvtColor(heatmap, cv2.COLOR_BGR2RGB) / 255.0

        overlay = np.clip(0.6 * img_vis + 0.4 * heatmap, 0, 1)
        frame = (overlay * 255).astype(np.uint8)

        text = f"{view_name.upper()} | Score: {prob:.4f}"

        h, w = frame.shape[:2]
        
        font = cv2.FONT_HERSHEY_SIMPLEX
        font_scale = 0.5     # mặc định nhỏ gọn
        thickness = 1
        
        # đo kích thước text
        (text_w, text_h), _ = cv2.getTextSize(text, font, font_scale, thickness)
        
        # nếu text dài hơn ảnh -> co lại vừa khung
        if text_w > w - 20:
            font_scale *= (w - 20) / text_w
            (text_w, text_h), _ = cv2.getTextSize(text, font, font_scale, thickness)
        
        # vẽ text (luôn nằm trong ảnh)
        cv2.putText(
            frame,
            text,
            (10, 10 + text_h),
            font,
            font_scale,
            (255, 255, 255),
            thickness,
            cv2.LINE_AA
        )

        # Lấy tên file gốc làm tên file PNG heatmap
        slice_name = os.path.splitext(os.path.basename(image_path))[0]
        save_path = os.path.join(patient_dir, f"{slice_name}_cam.png")
        Image.fromarray(frame).save(save_path)

    print(f"[GradCAM] Saved heatmaps -> {save_dir}")

In [ ]:
args = get_args()
set_seed(args.seed)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# Data transforms
transform = Compose([
    RandomAffine(degrees=(-5, 5), 
                 translate=(0.05, 0.05), 
                 scale=(0.85, 1.15), 
                 shear=5),
    ColorJitter(brightness=(0.9, 1.1)),
    Resize((args.image_size, args.image_size)),
    ToTensor(),
    Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

# Load full dataset
print("Loading dataset...")
full_dataset = OASIS2DDataset(root=args.root, transform=transform, view=VIEWS_TO_USE)

print(f"Total samples: {len(full_dataset)}")
print(f"Unique patients: {len(full_dataset.patients)}")
print(f"Class distribution: {Counter([full_dataset.labels[i] for i in range(len(full_dataset))])}")

# Get patient-level split
indices = list(full_dataset.patients.keys())
labels = [full_dataset.patients[p] for p in indices]

indices = np.array(indices)
labels = np.array(labels)

# 90-10 train-test split
train_val_patients, test_patients, train_val_labels, test_labels = train_test_split(
    indices, labels,
    test_size=0.1,
    stratify=labels,
    random_state=args.seed
)

# Convert patient IDs to image indices
train_val_idx = [
    i for i, pid in enumerate(full_dataset.id_patient)
    if pid in train_val_patients
]

test_idx = [
    i for i, pid in enumerate(full_dataset.id_patient)
    if pid in test_patients
]

print(f"Train+Val patients: {len(train_val_patients)} ({len(train_val_idx)} images)")
print(f"Test patients: {len(test_patients)} ({len(test_idx)} images)")

# Create output directories
os.makedirs(args.logging, exist_ok=True)
os.makedirs(args.trained_models, exist_ok=True)

writer = SummaryWriter(args.logging)

# K-Fold cross-validation
skf = StratifiedKFold(
    n_splits=args.folds,
    shuffle=True,
    random_state=args.seed
)

fold_results = []

for fold, (train_p_rel, val_p_rel) in enumerate(skf.split(train_val_patients, train_val_labels)):

    print(f"\n{'='*60}")
    print(f"FOLD {fold+1}/{args.folds}")
    print(f"{'='*60}")

    train_patients = set(train_val_patients[train_p_rel])
    val_patients = set(train_val_patients[val_p_rel])
    
    # Convert to image indices
    train_idx = [
        i for i, pid in enumerate(full_dataset.id_patient)
        if pid in train_patients
    ]
    val_idx = [
        i for i, pid in enumerate(full_dataset.id_patient)
        if pid in val_patients
    ]

    train_dataset = Subset(full_dataset, train_idx)
    val_dataset = Subset(full_dataset, val_idx)

    train_loader = DataLoader(
        train_dataset,
        batch_size=args.batch_size,
        shuffle=True,
        num_workers=2,
        drop_last=True
    )

    val_loader = DataLoader(
        val_dataset,
        batch_size=args.batch_size,
        shuffle=False,
        num_workers=2,
        drop_last=False
    )

    # Get class weights
    train_labels_fold = [full_dataset.labels[i] for i in train_idx]
    counter = Counter(train_labels_fold)
    total = sum(counter.values())
    num_classes = 2
    
    class_weights = []
    for c in range(num_classes):
        weight = total / (num_classes * counter[c])
        class_weights.append(weight)
    
    class_weights = torch.tensor(class_weights, dtype=torch.float).to(device)
    print(f"Class weights: {class_weights.cpu().numpy()}")

    # Create model
    if args.model == "mobilenet":
        model = MyMobileNetMultiAttention(num_classes=2, embed_dim=256).to(device)
    else:
        model = MyDenseNet121(num_classes=2, use_attention=True).to(device)

    print(f"Model: {args.model}")
    
    # Freeze backbone if needed
    for name, param in model.named_parameters():
        if any(layer in name for layer in TRAINABLE_LAYERS):
            param.requires_grad = True
        else:
            param.requires_grad = False

    total_params = sum(p.numel() for p in model.parameters())
    trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"Total parameters: {total_params:,}")
    print(f"Trainable parameters: {trainable_params:,}")

    # Loss & Optimizer
    alpha = class_weights
    criterion = FocalLoss(alpha=alpha, gamma=2)
    
    optimizer = torch.optim.Adam(
        filter(lambda p: p.requires_grad, model.parameters()),
        lr=args.learning_rate,
        weight_decay=args.weight_decay
    )

    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer,
        T_max=args.epochs
    )

    # Training
    best_acc = 0.0
    best_score = 0.0
    best_epoch = 0
    es_counter = 0
    
    epoch_train_losses = []
    epoch_val_accs = []
    epoch_val_aucs = []
    epoch_val_f1s = []
    
    for epoch in range(args.epochs):
        # Train
        model.train()
        running_loss = 0.0
        
        pbar = tqdm(train_loader, desc=f"Fold {fold+1} Epoch {epoch+1}/{args.epochs} [Train]", colour="green")
        
        for batch_idx, (images, labels_batch, _) in enumerate(pbar):
            images = images.to(device)
            labels_batch = labels_batch.to(device)
            
            outputs = model(images)
            loss = criterion(outputs, labels_batch)
            
            running_loss += loss.item()
            
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            
            pbar.set_postfix({'loss': f'{loss.item():.4f}'})
            
            global_step = fold * args.epochs * len(train_loader) + epoch * len(train_loader) + batch_idx
            writer.add_scalar("Train/Loss", loss.item(), global_step)
        
        avg_loss = running_loss / len(train_loader)
        epoch_train_losses.append(avg_loss)
        writer.add_scalar(f"Fold_{fold+1}/Train_Loss", avg_loss, epoch)
        
        # Validate
        model.eval()
        pat_probs = {}
        pat_labels = {}
        
        with torch.no_grad():
            for images, labels_batch, patient_ids in val_loader:
                images = images.to(device)
                
                outputs = model(images)
                probs = torch.softmax(outputs, dim=1)[:, 1]
                
                for pid, prob, lbl in zip(patient_ids, probs.cpu().tolist(), labels_batch.tolist()):
                    if pid not in pat_probs:
                        pat_probs[pid] = []
                        pat_labels[pid] = lbl
                    pat_probs[pid].append(prob)
        
        # Aggregate by patient
        all_probs = [np.mean(pat_probs[pid]) for pid in pat_probs]
        all_labels = [pat_labels[pid] for pid in pat_probs]
        all_preds = [1 if p >= 0.5 else 0 for p in all_probs]
        
        val_acc = accuracy_score(all_labels, all_preds)
        val_precision = precision_score(all_labels, all_preds, zero_division=0)
        val_recall = recall_score(all_labels, all_preds, zero_division=0)
        val_f1 = f1_score(all_labels, all_preds, zero_division=0)
        
        try:
            val_auc = roc_auc_score(all_labels, all_probs)
        except ValueError:
            val_auc = 0.0
        
        score = ACC_RATIO * val_acc + (1 - ACC_RATIO) * val_recall
        
        epoch_val_accs.append(val_acc)
        epoch_val_aucs.append(val_auc)
        epoch_val_f1s.append(val_f1)
        
        writer.add_scalar(f"Fold_{fold+1}/Val_Acc", val_acc, epoch)
        writer.add_scalar(f"Fold_{fold+1}/Val_F1", val_f1, epoch)
        writer.add_scalar(f"Fold_{fold+1}/Val_AUC", val_auc, epoch)
        
        print(f"Val | Acc: {val_acc:.4f} | F1: {val_f1:.4f} | AUC: {val_auc:.4f}")
        
        scheduler.step()
        
        # Save best model
        if score > best_score:
            best_score = score
            best_acc = val_acc
            best_epoch = epoch + 1
            es_counter = 0
            
            torch.save({
                'epoch': epoch + 1,
                'model': model.state_dict(),
                'optimizer': optimizer.state_dict(),
                'best_acc': best_acc,
            }, f"{args.trained_models}/fold_{fold+1}_best.pt")
        else:
            es_counter += 1
            if es_counter >= args.early_stopping:
                print(f"Early stopping at epoch {epoch+1}")
                break
        
        # Save last
        torch.save({
            'epoch': epoch + 1,
            'model': model.state_dict(),
        }, f"{args.trained_models}/fold_{fold+1}_last.pt")
    
    fold_results.append({
        'fold': fold + 1,
        'best_epoch': best_epoch,
        'best_acc': best_acc,
        'best_score': best_score,
    })
    
    # Plot fold metrics
    plot_metrics(
        writer=writer,
        train_losses=epoch_train_losses,
        val_acc=epoch_val_accs,
        val_auc=epoch_val_aucs,
        val_labels=all_labels,
        val_probs=all_probs,
        class_names=full_dataset.categories if hasattr(full_dataset, 'categories') else ['CN', 'AD'],
        epoch=fold,
        tag=f"Fold_{fold+1}",
        save_dir=OUTPUT
    )

# Summary
print(f"\n{'='*60}")
print("FINAL RESULTS")
print(f"{'='*60}")
for result in fold_results:
    print(f"Fold {result['fold']}: Best Epoch {result['best_epoch']}, Acc {result['best_acc']:.4f}")

best_fold = max(fold_results, key=lambda x: x['best_acc'])
print(f"\nBest Fold: {best_fold['fold']} with accuracy {best_fold['best_acc']:.4f}")

writer.close()



========== FOLD 1/5 ==========


  0%|          | 0/258 [00:00<?, ?it/s]

Fold 1 | Epoch 1 | Accuracy: 0.5349 | Precision: 0.2973 | Recall: 0.7333 | F1: 0.4231 | AUC: 0.6767676767676768


  0%|          | 0/258 [00:00<?, ?it/s]

Fold 1 | Epoch 2 | Accuracy: 0.2403 | Precision: 0.2344 | Recall: 1.0000 | F1: 0.3797 | AUC: 0.6225589225589225


...

Fold 5 | Epoch 28 | Accuracy: 0.7287 | Precision: 0.3636 | Recall: 0.2759 | F1: 0.3137 | AUC: 0.603448275862069


  0%|          | 0/258 [00:00<?, ?it/s]

Fold 5 | Epoch 29 | Accuracy: 0.7597 | Precision: 0.4167 | Recall: 0.1724 | F1: 0.2439 | AUC: 0.6686206896551724


  0%|          | 0/258 [00:00<?, ?it/s]

Fold 5 | Epoch 30 | Accuracy: 0.7364 | Precision: 0.3077 | Recall: 0.1379 | F1: 0.1905 | AUC: 0.6355172413793103
  [Early Stopping] No improvement for 20 epochs. Stop fold 5.

===== BEST RESULT FOLD 5 =====
Best Epoch : 10
Accuracy   : 0.6512
Precision  : 0.3571
Recall     : 0.6897
F1         : 0.4706
AUC        : 0.6907
Best Confusion Matrix:
 [[64 36]
 [ 9 20]]

========== BEST MODEL (Fold 4) ==========
Accuracy   : 0.7209
Precision  : 0.4118
Recall     : 0.4667
F1         : 0.4375


In [ ]:
# Load best model and visualize heatmaps
best_fold_num = best_fold['fold']
ckpt_path = f"{args.trained_models}/fold_{best_fold_num}_best.pt"

if os.path.exists(ckpt_path):
    model = MyDenseNet121(num_classes=2, use_attention=True).to(device)
    ckpt = torch.load(ckpt_path, map_location=device)
    model.load_state_dict(ckpt['model'])
    model.eval()
    
    print(f"Loaded checkpoint from fold {best_fold_num}")
    print(f"Checkpoint epoch: {ckpt['epoch']}")
    print(f"Best accuracy: {ckpt['best_acc']:.4f}")
else:
    print(f"Checkpoint not found: {ckpt_path}")


NameError: name 'kaggle' is not defined

# Test.py

In [ ]:
print("=== Test patients summary ===")

print("Total test patients:", len(test_patients))
print("Patient IDs:", list(test_patients))

print("\nPatient -> Class:")
for p in test_patients:
    label = full_dataset.patients[p]
    class_name = full_dataset.categories[label]
    print(p, "->", class_name)

print("=== End test patients summary ===")

In [ ]:
# Test on held-out test set
print("\n" + "="*60)
print("TESTING ON HELD-OUT SET")
print("="*60)

test_dataset = Subset(full_dataset, test_idx)
test_loader = DataLoader(
    test_dataset,
    batch_size=args.batch_size,
    shuffle=False,
    num_workers=2,
)

model.eval()
pat_probs = {}
pat_labels = {}

with torch.no_grad():
    for images, labels_batch, patient_ids in test_loader:
        images = images.to(device)
        
        outputs = model(images)
        probs = torch.softmax(outputs, dim=1)[:, 1]
        
        for pid, prob, lbl in zip(patient_ids, probs.cpu().tolist(), labels_batch.tolist()):
            if pid not in pat_probs:
                pat_probs[pid] = []
                pat_labels[pid] = lbl
            pat_probs[pid].append(prob)

# Aggregate by patient
all_probs = [np.mean(pat_probs[pid]) for pid in pat_probs]
all_labels = [pat_labels[pid] for pid in pat_probs]
all_preds = [1 if p >= 0.5 else 0 for p in all_probs]

test_acc = accuracy_score(all_labels, all_preds)
test_precision = precision_score(all_labels, all_preds, zero_division=0)
test_recall = recall_score(all_labels, all_preds, zero_division=0)
test_f1 = f1_score(all_labels, all_preds, zero_division=0)

try:
    test_auc = roc_auc_score(all_labels, all_probs)
except ValueError:
    test_auc = 0.0

print(f"Test Accuracy : {test_acc:.4f}")
print(f"Test Precision: {test_precision:.4f}")
print(f"Test Recall   : {test_recall:.4f}")
print(f"Test F1       : {test_f1:.4f}")
print(f"Test AUC      : {test_auc:.4f}")

# Plot test metrics
plot_metrics(
    writer=writer,
    train_losses=[0.0],
    val_acc=[test_acc],
    val_auc=[test_auc],
    val_labels=all_labels,
    val_probs=all_probs,
    class_names=['CN', 'AD'],
    epoch=0,
    tag="Test_Final",
    save_dir=OUTPUT
)

writer.close()
print("\nTraining complete!")
